In [1]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Préparation de données

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
VAL_SPLIT = 0.2
SEED = 42

In [3]:
train_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

eval_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

In [4]:
# Split train / validation
source = datasets.ImageFolder('data/Training')
class_names = source.classes
num_classes = len(class_names)

val_size = int(VAL_SPLIT * len(source))
train_size = len(source) - val_size
train_indices, val_indices = random_split(
    range(len(source)), [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

In [5]:
# Dataset custom pour appliquer des transforms differents sur chaque subset
class SubsetWithTransform(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        return self.transform(img), label

In [6]:
train_dataset = SubsetWithTransform(source, train_indices, train_tf)
val_dataset   = SubsetWithTransform(source, val_indices,   eval_tf)
test_dataset  = datasets.ImageFolder('data/Testing', transform=eval_tf)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Classes ({num_classes}): {class_names}")

Train: 4480 | Val: 1120 | Test: 1600
Classes (4): ['glioma', 'meningioma', 'notumor', 'pituitary']


## Nettoyage 

In [7]:
from PIL import Image
import os

def check_images(folder):
    broken = []
    for root, _, files in os.walk(folder):
        for f in files:
            path = os.path.join(root, f)
            try:
                img = Image.open(path)
                img.verify()
            except:
                broken.append(path)
    return broken

print(check_images('data/Training'))
print(check_images('data/Testing'))

[]
[]


aucune image corrompue détectée :)

# Entrainement du modele